# Dynamic Programming — Coding Interview Prep

Climbing stairs, coin change, LCS, knapsack, and edit distance.

## 1. Climbing Stairs & Fibonacci — 1D DP

In [ ]:
from functools import lru_cache

# Tabulation (bottom-up)
def climb_stairs(n):
    if n <= 2: return n
    dp = [0] * (n + 1)
    dp[1], dp[2] = 1, 2
    for i in range(3, n + 1):
        dp[i] = dp[i-1] + dp[i-2]
    return dp[n]

# Space-optimized O(1)
def climb_stairs_opt(n):
    a, b = 1, 2
    for _ in range(n - 2):
        a, b = b, a + b
    return b if n >= 2 else n

assert [climb_stairs(i) for i in range(1, 8)] == [1, 2, 3, 5, 8, 13, 21]

# House robber — max sum of non-adjacent elements
def rob(nums):
    if not nums: return 0
    prev2, prev1 = 0, 0
    for n in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + n)
    return prev1

assert rob([2, 7, 9, 3, 1]) == 12
assert rob([2, 1, 1, 2]) == 4
print("1D DP: all tests passed")

## 2. Coin Change — Unbounded Knapsack Pattern

In [ ]:
# Minimum coins
def coin_change(coins, amount):
    dp = [float('inf')] * (amount + 1)
    dp[0] = 0
    for c in coins:
        for a in range(c, amount + 1):
            dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] != float('inf') else -1

assert coin_change([1, 5, 11], 15) == 3   # 11+1+1+1... no: 5+5+5=3
assert coin_change([2], 3) == -1
assert coin_change([1, 2, 5], 11) == 3    # 5+5+1

# Number of ways to make change
def coin_ways(coins, amount):
    dp = [0] * (amount + 1)
    dp[0] = 1
    for c in coins:
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]

assert coin_ways([1, 2, 5], 5) == 4  # (5),(2+2+1),(2+1+1+1),(1*5)
print("Coin change: all tests passed")

## 3. 0/1 Knapsack

In [ ]:
def knapsack_01(weights, values, capacity):
    n = len(weights)
    # dp[i][w] = max value using first i items with capacity w
    dp = [[0] * (capacity + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for w in range(capacity + 1):
            dp[i][w] = dp[i-1][w]  # don't take item i
            if weights[i-1] <= w:
                dp[i][w] = max(dp[i][w], dp[i-1][w - weights[i-1]] + values[i-1])
    return dp[n][capacity]

# Space-optimized: iterate w in reverse
def knapsack_01_opt(weights, values, capacity):
    dp = [0] * (capacity + 1)
    for w, v in zip(weights, values):
        for c in range(capacity, w - 1, -1):  # reverse to avoid reuse
            dp[c] = max(dp[c], dp[c - w] + v)
    return dp[capacity]

weights = [1, 3, 4, 5]
values  = [1, 4, 5, 7]
capacity = 7
assert knapsack_01(weights, values, capacity) == 9
assert knapsack_01_opt(weights, values, capacity) == 9

# Partition equal subset sum (variant)
def can_partition(nums):
    total = sum(nums)
    if total % 2: return False
    target = total // 2
    dp = {0}
    for n in nums:
        dp |= {s + n for s in dp}
    return target in dp

assert can_partition([1, 5, 11, 5]) == True
assert can_partition([1, 2, 3, 5]) == False
print("Knapsack: all tests passed")

## 4. Longest Common Subsequence (LCS)

In [ ]:
def lcs(text1, text2):
    m, n = len(text1), len(text2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if text1[i-1] == text2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

assert lcs("abcde", "ace") == 3
assert lcs("abc", "abc") == 3
assert lcs("abc", "def") == 0

# Longest Increasing Subsequence O(n log n)
import bisect
def lis(nums):
    tails = []
    for n in nums:
        idx = bisect.bisect_left(tails, n)
        if idx == len(tails): tails.append(n)
        else: tails[idx] = n
    return len(tails)

assert lis([10, 9, 2, 5, 3, 7, 101, 18]) == 4  # [2,3,7,101]
assert lis([0, 1, 0, 3, 2, 3]) == 4
print("LCS/LIS: all tests passed")

## 5. Edit Distance (Levenshtein)

In [ ]:
def edit_distance(word1, word2):
    m, n = len(word1), len(word2)
    dp = list(range(n + 1))  # space-optimized: 1D
    for i in range(1, m + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            if word1[i-1] == word2[j-1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j-1], dp[j])  # replace, insert, delete
            prev = temp
    return dp[n]

assert edit_distance("horse", "ros") == 3
assert edit_distance("intention", "execution") == 5
assert edit_distance("", "abc") == 3
print("Edit distance: all tests passed")

## 6. 2D Grid DP

In [ ]:
# Unique paths
def unique_paths(m, n):
    dp = [1] * n
    for _ in range(1, m):
        for j in range(1, n):
            dp[j] += dp[j-1]
    return dp[-1]

assert unique_paths(3, 7) == 28
assert unique_paths(3, 2) == 3

# Minimum path sum
def min_path_sum(grid):
    m, n = len(grid), len(grid[0])
    for i in range(m):
        for j in range(n):
            if i == 0 and j == 0: continue
            top = grid[i-1][j] if i > 0 else float('inf')
            left = grid[i][j-1] if j > 0 else float('inf')
            grid[i][j] += min(top, left)
    return grid[-1][-1]

assert min_path_sum([[1,3,1],[1,5,1],[4,2,1]]) == 7
print("2D Grid DP: all tests passed")

## DP Framework

**4 steps to solve any DP:**
1. **Define state**: dp[i] or dp[i][j] = ?
2. **Recurrence**: how dp[i] relates to smaller subproblems
3. **Base cases**: dp[0] or dp[0][0]
4. **Order**: ensure subproblems solved before using them

| Pattern | Recurrence | Example |
|---------|-----------|--------|
| 1D linear | dp[i] = f(dp[i-1]) | Climbing stairs |
| 1D non-adjacent | dp[i] = max(dp[i-1], dp[i-2]+val) | House robber |
| Unbounded knapsack | dp[a] = min(dp[a], dp[a-c]+1) | Coin change |
| 0/1 knapsack | dp[i][w] = max(skip, take) | Knapsack |
| 2-sequence | dp[i][j] = match or max(left,up) | LCS, edit distance |